# Deep Learning Regression — *placeholder title*

In notebook 01 we predicted `y` from a single feature `x` with a
straight line. Some relationships are **non-linear**: the right
amount of one input is good, twice as much is bad. A
**neural network** is the natural tool for those.

## The story, extended

Each observation now has **four features** (we will call them
`f1, f2, f3, f4`) and one target `y`. Some features interact
non-linearly. The skill's glossary cell above tells you what each
feature and the target mean in your field.

> **Glossary cell — replaced by the adaptation skill.**
>
> When you run the `adapt-notebook` skill, this cell is replaced with a small
> table mapping vocabulary from your field to the ML terms used below.
> If you have not run the skill, you can safely ignore this cell.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
rng = np.random.default_rng(0)

## 1. The data

Each row is one past observation. The four columns are the
**features** `f1, f2, f3, f4` (here scaled to 0–1) and the column
we want to predict is the **target** `y`.

### From one feature to many

We now have a feature *vector* per observation, not a single number.
The model still produces one **prediction** per observation; it just
has more inputs to combine.

### In your field

The adaptation skill rewrites this cell to say what each of the
four features `f1, f2, f3, f4` represents in your work — four
quantities you would actually measure or look up about a single
case — and what `y` is when those four come together.

In [ ]:
N = 800
X = rng.uniform(0, 1, size=(N, 4)).astype(np.float32)
f1, f2, f3, f4 = X[:, 0], X[:, 1], X[:, 2], X[:, 3]

# Non-linear ground truth: a smooth function with interactions.
y_true = (
    5.0
    + 3.0 * np.sin(2 * np.pi * f1)
    + 4.0 * (f2 * f3)
    - 6.0 * (f4 - 0.4) ** 2
    + rng.normal(0, 0.3, size=N).astype(np.float32)
)
y = y_true.astype(np.float32)
X.shape, y.shape

A linear model cannot capture this — `y` is not a straight-line
function of any single feature. We need a model that can learn
curves and interactions between features.

### Go deeper with an LLM (optional — skip if you already know this)

The adaptation skill replaces this block with 2–4 copy-pasteable prompts
tailored to your background. Paste them into ChatGPT / Claude / Mistral /
your preferred model, explore the idea, then come back here and continue.

## 2. The model — a small neural network

A **neural network** stacks layers of simple computations.
Each **hidden layer** computes `h = activation(W · x + b)`, where
`W` and `b` are learnable **weights** and **bias** and the
**activation function** is a **non-linearity** such as **ReLU**
($\text{ReLU}(z) = \max(0, z)$). Without an activation function,
stacking layers would still only give you a linear model.

Our network has 4 inputs → 16 hidden units → 16 hidden units → 1
output (the predicted target).

### In your field

The adaptation skill rewrites this cell to describe the network
in your own terms — what each hidden layer is doing as it mixes
the four field-specific inputs, and why a stack of small
non-linear steps lets the model bend in ways a single rule of
thumb cannot.

### Worked example

Each hidden unit can be thought of as one specialist paying
attention to a particular combination of features. The next layer
combines the specialists' opinions into a final prediction. The
skill will rewrite this analogy in your field's terms.

In [ ]:
class RegressionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(4, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.layers(x).squeeze(-1)

net = RegressionNet()
sum(p.numel() for p in net.parameters())   # total parameters

## 3. Training — forward pass, loss, backward pass

Training one **batch** of data has three steps:

1. **Forward pass** — feed the features through the network to get
   predictions.
2. Compute the **loss** (MSE again) by comparing predictions to
   targets.
3. **Backward pass** (also called **backpropagation**) — compute
   the **gradient** of the loss with respect to every weight, then
   ask the **optimizer** to nudge the weights one step.

We repeat for many **epochs** until the loss stops improving.

### In your field

The adaptation skill rewrites this cell to put the forward pass,
loss, backward pass, and optimizer step into your own working
rhythm — the equivalent of "try, check the gap, adjust, try
again" you already use in your day job, just done thousands of
times automatically.

In [ ]:
# Train/test split.
idx = rng.permutation(N)
n_train = int(0.8 * N)
train_idx, test_idx = idx[:n_train], idx[n_train:]

X_train = torch.tensor(X[train_idx])
y_train = torch.tensor(y[train_idx])
X_test  = torch.tensor(X[test_idx])
y_test  = torch.tensor(y[test_idx])

In [ ]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(net.parameters(), lr=1e-2)

epochs = 300
batch_size = 64
history = []

for epoch in range(epochs):
    perm = torch.randperm(X_train.shape[0])
    epoch_loss = 0.0
    for start in range(0, X_train.shape[0], batch_size):
        b = perm[start:start + batch_size]
        pred = net(X_train[b])                  # forward pass
        loss = loss_fn(pred, y_train[b])
        optimizer.zero_grad()
        loss.backward()                          # backward pass
        optimizer.step()
        epoch_loss += float(loss) * b.shape[0]
    history.append(epoch_loss / X_train.shape[0])

print(f"final train MSE = {history[-1]:.3f}")

In [ ]:
# The loss curve: each point is the average MSE for one full epoch.
plt.figure(figsize=(6, 4))
plt.plot(history)
plt.xlabel("epoch")
plt.ylabel("MSE (loss)")
plt.title("Training the regression network")
plt.grid(True, alpha=0.3)
plt.show()

### Go deeper with an LLM (optional — skip if you already know this)

The adaptation skill replaces this block with 2–4 copy-pasteable prompts
tailored to your background. Paste them into ChatGPT / Claude / Mistral /
your preferred model, explore the idea, then come back here and continue.

## 4. Did it actually learn?

We now look at the **test data** the network has never seen.
If the test MSE is close to the training MSE, the model has learned
a generalisable rule, not just memorised. If the test MSE is much
higher, we have **overfitting** — the same enemy as in notebook 01,
only sneakier in deep models with many parameters.

### In your field

The adaptation skill rewrites this cell to describe — in your
own working language — what overfitting looks like on *your*
data when the network has many parameters and only a modest pile
of past observations to learn from.

In [ ]:
with torch.no_grad():
    test_pred = net(X_test)
    test_mse = float(loss_fn(test_pred, y_test))
print(f"test MSE = {test_mse:.3f}")

plt.figure(figsize=(5, 5))
plt.scatter(y_test.numpy(), test_pred.numpy(), alpha=0.6)
lims = [float(y_test.min()) - 0.5, float(y_test.max()) + 0.5]
plt.plot(lims, lims, "k--", alpha=0.5)
plt.xlabel("true target y")
plt.ylabel("predicted target y")
plt.title("Predictions vs truth (test set)")
plt.grid(True, alpha=0.3)
plt.show()

Points clustered along the diagonal mean the network has captured
the non-linear interactions between features. A linear regression
on the same data would leave a much messier scatter.

### Go deeper with an LLM (optional — skip if you already know this)

The adaptation skill replaces this block with 2–4 copy-pasteable prompts
tailored to your background. Paste them into ChatGPT / Claude / Mistral /
your preferred model, explore the idea, then come back here and continue.

## Try this yourself

1. Reduce the network to a single `nn.Linear(4, 1)` (no hidden
   layer, no activation function). How much worse is the test MSE?
   Why?
2. Increase the learning rate to `0.5`. What happens to the loss
   curve?
3. Replace `nn.ReLU()` with `nn.Tanh()`. Does it train as well?

## Recap — vocabulary you now own

On top of notebook 01: **neural network**, **hidden layer**,
**activation function**, **ReLU**, **forward pass**,
**backward pass**, **backpropagation**, **optimizer**, **batch**,
**non-linearity**.

The training loop — forward, loss, backward, step — is the same
loop that powers every modern model, from a 4-input regressor like
this one to a 100-billion-parameter language model.